<a href="https://colab.research.google.com/github/cto-school/mathematics-for-machine-learning/blob/main/14-model-evaluation-and-regularization/14a_evaluation_metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 14a — Model Evaluation Metrics (Beyond Accuracy)

> **The big idea.** Training a model is only half the job. The other half is
> **measuring honestly how good it is**. A single number like "accuracy" can be
> deeply misleading. This notebook builds the metrics that working machine
> learning actually relies on — the **confusion matrix**, **precision**,
> **recall**, **F1**, the **threshold trade-off**, regression metrics
> (**MSE / RMSE / MAE / R²**), and the gold standard for choosing settings:
> **cross-validation** — all from scratch in NumPy.

By the end you will be able to:

- explain *why* accuracy lies on **imbalanced** data;
- build a **confusion matrix** (TP / FP / FN / TN) and read it;
- compute **precision**, **recall**, and **F1**, and say when each one matters;
- move the **decision threshold** and watch precision trade against recall (and
  sketch an **ROC**-style curve);
- recap the regression metrics **MSE / RMSE / MAE / R²**;
- split data into **train / validation / test** and run **k-fold
  cross-validation** to pick a hyperparameter.

Run every code cell with **Shift + Enter**. Edit and re-run freely.

## 0. Setup

We use only **NumPy** and **Matplotlib**. We fix the random seed with
`np.random.default_rng(0)` so every run looks identical.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)                   # reproducible randomness
np.set_printoptions(precision=4, suppress=True)  # tidy array printing

## 1. Why accuracy misleads on imbalanced classes

Recall **accuracy** = fraction of predictions that are correct,

$$\text{accuracy} = \frac{1}{n}\sum_{i=1}^{n}\mathbf{1}\{\hat{y}_i = y_i\}.$$

It is a fine summary when the two classes are roughly **balanced**. But suppose
we screen for a rare disease that only **1 in 100** people have. Consider the
"model" that ignores the input entirely and always predicts **healthy**.

In [ ]:
# 1000 patients; only 1% are actually sick (label 1)
n = 1000
y_true = np.zeros(n, dtype=int)
y_true[:10] = 1                      # 10 sick, 990 healthy
rng.shuffle(y_true)

# the lazy classifier: ALWAYS predict 0 ("healthy"), no matter what
y_pred_lazy = np.zeros(n, dtype=int)

def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

print("accuracy of 'always healthy':", accuracy(y_true, y_pred_lazy))

**99% accuracy — and the model is useless.** It never catches a single sick
patient, which is the whole point of the test. Accuracy hid the failure because
it lumps the rare, important class in with the common one. We need metrics that
look at the **types** of mistakes separately. That is the confusion matrix.

## 2. The confusion matrix (TP / FP / FN / TN)

For binary classification (positive = 1, negative = 0) every prediction falls
into one of **four** boxes:

|                       | actual **1** (positive) | actual **0** (negative) |
|-----------------------|-------------------------|-------------------------|
| **predicted 1**       | **TP** true positive    | **FP** false positive   |
| **predicted 0**       | **FN** false negative   | **TN** true negative    |

- **TP** — correctly flagged a positive.
- **FP** — false alarm: said positive, was actually negative.
- **FN** — a miss: said negative, was actually positive.
- **TN** — correctly left a negative alone.

Everything else (accuracy, precision, recall, ...) is just arithmetic on these
four counts. Let's count them **from scratch**.

In [ ]:
def confusion_counts(y_true, y_pred):
    """Return TP, FP, FN, TN for binary labels in {0, 1}."""
    tp = int(np.sum((y_pred == 1) & (y_true == 1)))   # said 1, was 1
    fp = int(np.sum((y_pred == 1) & (y_true == 0)))   # said 1, was 0
    fn = int(np.sum((y_pred == 0) & (y_true == 1)))   # said 0, was 1
    tn = int(np.sum((y_pred == 0) & (y_true == 0)))   # said 0, was 0
    return tp, fp, fn, tn

tp, fp, fn, tn = confusion_counts(y_true, y_pred_lazy)
print(f"TP={tp}  FP={fp}  FN={fn}  TN={tn}")
# The lazy model: TP=0 (caught nobody), FN=10 (missed every sick patient).

Let's also build a slightly smarter (but imperfect) classifier so the matrix has
interesting numbers, then **visualize** it with `imshow`.

In [ ]:
# a noisy but real classifier: catches most sick people, with some false alarms
y_pred = y_true.copy()
flip = rng.random(n) < 0.05            # randomly flip ~5% of labels as "errors"
y_pred[flip] = 1 - y_pred[flip]

tp, fp, fn, tn = confusion_counts(y_true, y_pred)
print(f"TP={tp}  FP={fp}  FN={fn}  TN={tn}")

In [ ]:
# arrange the four counts into a 2x2 matrix and draw it
# rows = predicted (1 then 0), columns = actual (1 then 0)
M = np.array([[tp, fp],
              [fn, tn]])

fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(M, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["actual 1", "actual 0"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["pred 1", "pred 0"])
for i in range(2):                     # write the count inside each cell
    for j in range(2):
        ax.text(j, i, M[i, j], ha="center", va="center",
                color="black", fontsize=14)
ax.set_title("Confusion matrix")
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

## 3. Precision, recall, and F1

From the four counts we build the three metrics you will use constantly.

**Precision** — of everything we *flagged positive*, what fraction really was?
It punishes **false alarms (FP)**.

$$\text{precision} = \frac{TP}{TP + FP}.$$

**Recall** (a.k.a. *sensitivity*) — of all the *actual positives*, what fraction
did we *catch*? It punishes **misses (FN)**.

$$\text{recall} = \frac{TP}{TP + FN}.$$

**F1 score** — the **harmonic mean** of the two, a single number that is high
only when *both* are high:

$$F_1 = 2 \cdot \frac{\text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}.$$

In [ ]:
def precision(y_true, y_pred):
    tp, fp, fn, tn = confusion_counts(y_true, y_pred)
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0

def recall(y_true, y_pred):
    tp, fp, fn, tn = confusion_counts(y_true, y_pred)
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0

def f1_score(y_true, y_pred):
    p, r = precision(y_true, y_pred), recall(y_true, y_pred)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

print("precision:", round(precision(y_true, y_pred), 3))
print("recall   :", round(recall(y_true, y_pred), 3))
print("F1       :", round(f1_score(y_true, y_pred), 3))

Now compare against the lazy "always healthy" model. Its accuracy was 99%, but:

In [ ]:
print("lazy model precision:", round(precision(y_true, y_pred_lazy), 3))
print("lazy model recall   :", round(recall(y_true, y_pred_lazy), 3))
print("lazy model F1       :", round(f1_score(y_true, y_pred_lazy), 3))
# recall = 0: it caught NONE of the sick. F1 = 0. The 99% accuracy was a mirage.

### When does each one matter?

- **Recall matters most** when a **miss is expensive**: disease screening, fraud
  detection, "is there a pedestrian in front of the car?" You'd rather raise a
  few false alarms than let a true positive slip through.
- **Precision matters most** when a **false alarm is expensive**: flagging an
  email as spam (don't bury a real email), recommending an aggressive treatment.
- **F1** is the go-to single number when you care about **both** and the classes
  are imbalanced (so accuracy is untrustworthy).

---
## ✍️ Exercise 1 (solution included)

A model is evaluated and produces these counts: **TP = 40, FP = 10, FN = 20,
TN = 930**. By hand-feeding these into the formulas (no data needed), compute the
**accuracy, precision, recall, and F1**. Which is highest, and why is accuracy so
optimistic here?

**Solution:**

In [ ]:
tp, fp, fn, tn = 40, 10, 20, 930
total = tp + fp + fn + tn

acc = (tp + tn) / total
prec = tp / (tp + fp)
rec  = tp / (tp + fn)
f1   = 2 * prec * rec / (prec + rec)

print(f"accuracy : {acc:.3f}")
print(f"precision: {prec:.3f}")
print(f"recall   : {rec:.3f}")
print(f"F1       : {f1:.3f}")
# Accuracy (0.97) is highest because the 930 easy true-negatives dominate the
# count. Precision/recall ignore the TN box, so they reveal the model is far
# from perfect at the task that matters (catching positives).

## 4. The classification threshold trade-off

Most classifiers don't output 0/1 directly — they output a **score** or
**probability** $\hat{p} \in [0, 1]$. We turn that into a label by comparing it
to a **threshold** $t$:

$$\hat{y} = 1 \quad\text{if}\quad \hat{p} \ge t, \qquad \hat{y} = 0 \text{ otherwise.}$$

The default $t = 0.5$ is just a convention. **Moving $t$ trades precision against
recall:**

- **Lower $t$** → we flag more things positive → catch more true positives
  (**recall up**) but also more false alarms (**precision down**).
- **Raise $t$** → we are stricter → fewer false alarms (**precision up**) but we
  miss more (**recall down**).

Let's make scored data and sweep the threshold.

In [ ]:
# 400 samples; positives tend to get higher scores, but the two overlap
m = 400
labels = rng.integers(0, 2, size=m)                  # true 0/1 labels
scores = np.where(labels == 1,
                  rng.normal(0.65, 0.18, m),         # positives: higher scores
                  rng.normal(0.35, 0.18, m))         # negatives: lower scores
scores = np.clip(scores, 0, 1)

thresholds = np.linspace(0, 1, 101)
precs, recs = [], []
for t in thresholds:
    pred = (scores >= t).astype(int)
    precs.append(precision(labels, pred))
    recs.append(recall(labels, pred))
precs, recs = np.array(precs), np.array(recs)

In [ ]:
plt.plot(thresholds, precs, label="precision")
plt.plot(thresholds, recs,  label="recall")
plt.axvline(0.5, color="gray", ls="--", label="default t = 0.5")
plt.xlabel("threshold t"); plt.ylabel("score")
plt.title("Precision and recall vs decision threshold")
plt.legend(); plt.grid(True)
plt.show()
# As t rises (left to right): recall falls, precision rises. The classic tug-of-war.

### An ROC-style curve (optional but illuminating)

The **ROC curve** plots the **true-positive rate** (= recall) against the
**false-positive rate** as the threshold sweeps:

$$\text{TPR} = \frac{TP}{TP + FN}, \qquad \text{FPR} = \frac{FP}{FP + TN}.$$

A perfect classifier hugs the **top-left** corner; the diagonal is random
guessing. The **area under the curve (AUC)** summarizes it in one number (1.0 =
perfect, 0.5 = coin flip).

In [ ]:
tprs, fprs = [], []
for t in thresholds:
    pred = (scores >= t).astype(int)
    tp, fp, fn, tn = confusion_counts(labels, pred)
    tprs.append(tp / (tp + fn) if (tp + fn) > 0 else 0.0)
    fprs.append(fp / (fp + tn) if (fp + tn) > 0 else 0.0)
tprs, fprs = np.array(tprs), np.array(fprs)

# AUC via the trapezoidal rule (fprs run high->low as t increases, so negate)
auc = np.trapz(tprs[::-1], fprs[::-1])

plt.plot(fprs, tprs, "-o", markersize=3, label=f"ROC (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="random guessing")
plt.xlabel("false-positive rate"); plt.ylabel("true-positive rate (recall)")
plt.title("ROC curve")
plt.legend(); plt.grid(True)
plt.show()

## 5. Regression metrics: MSE, RMSE, MAE, R²

For **regression** the prediction is a number, so we measure *how far off* it is.

- **MSE** — mean squared error (what we usually minimize):
  $$\mathrm{MSE} = \frac{1}{n}\sum_i (\hat{y}_i - y_i)^2.$$
- **RMSE** — its square root, **back in the original units** (nicer to report):
  $$\mathrm{RMSE} = \sqrt{\mathrm{MSE}}.$$
- **MAE** — mean absolute error, more robust to outliers:
  $$\mathrm{MAE} = \frac{1}{n}\sum_i |\hat{y}_i - y_i|.$$
- **R²** (coefficient of determination) — the fraction of variance the model
  explains. **1.0 is perfect**; **0** means "no better than predicting the mean
  $\bar{y}$"; negative means *worse* than the mean.
  $$R^2 = 1 - \frac{\sum_i (\hat{y}_i - y_i)^2}{\sum_i (y_i - \bar{y})^2}.$$

In [ ]:
def mse(y_true, y_hat):
    return np.mean((y_hat - y_true)**2)

def rmse(y_true, y_hat):
    return np.sqrt(mse(y_true, y_hat))

def mae(y_true, y_hat):
    return np.mean(np.abs(y_hat - y_true))

def r2_score(y_true, y_hat):
    ss_res = np.sum((y_hat - y_true)**2)          # residual sum of squares
    ss_tot = np.sum((y_true - np.mean(y_true))**2)  # total variance of y
    return 1 - ss_res / ss_tot

In [ ]:
# toy regression: y = 3x + 2 + noise, fit a line, measure all four metrics
x = np.linspace(0, 10, 50)
y = 3 * x + 2 + rng.normal(0, 2.0, x.size)

coeffs = np.polyfit(x, y, deg=1)         # least-squares line
y_hat = np.polyval(coeffs, x)

print(f"MSE : {mse(y, y_hat):.4f}")
print(f"RMSE: {rmse(y, y_hat):.4f}   (same units as y)")
print(f"MAE : {mae(y, y_hat):.4f}")
print(f"R2  : {r2_score(y, y_hat):.4f}   (close to 1 = good fit)")

---
## ✍️ Exercise 2 (solution included)

Show that a model which **always predicts the mean** $\bar{y}$ gets $R^2 = 0$
exactly. Compute it on the data above to confirm, and explain in one line why
this is the natural "zero" of $R^2$.

**Solution:**

In [ ]:
y_mean_pred = np.full_like(y, np.mean(y))    # constant prediction = y-bar
print("R2 of mean-predictor:", round(r2_score(y, y_mean_pred), 6))
# It is 0 because then ss_res == ss_tot, so 1 - ss_res/ss_tot = 0. R2 measures
# improvement OVER just guessing the mean, so the mean-predictor scores exactly 0.

## 6. Validating properly: train / validation / test

In Chapter 9 we split data into **train** and **test**. But there is a subtle
trap: if we keep tweaking the model while watching the **test** error, we slowly
**leak** the test set into our decisions — the test score stops being honest.

The fix is a **three-way split**:

- **training set** — fit the model parameters $\theta$;
- **validation set** — choose **hyperparameters** (degree, $\lambda$, ...) and
  compare models;
- **test set** — locked in a vault, touched **once** at the very end.

Let's build the three-way split from scratch (shuffle, then slice).

In [ ]:
def train_val_test_split(X, y, val_frac=0.2, test_frac=0.2, rng=rng):
    n = X.shape[0]
    idx = np.arange(n)
    rng.shuffle(idx)                          # shuffle so order can't poison it

    n_val  = int(round(n * val_frac))
    n_test = int(round(n * test_frac))
    val_idx   = idx[:n_val]
    test_idx  = idx[n_val:n_val + n_test]
    train_idx = idx[n_val + n_test:]          # whatever remains -> train

    return (X[train_idx], X[val_idx], X[test_idx],
            y[train_idx], y[val_idx], y[test_idx])

# demo on a curve-plus-noise dataset
N = 60
xx = np.sort(rng.uniform(0, 5, N))
yy = np.sin(1.2 * xx) + 0.5 * xx + rng.normal(0, 0.35, N)
XX = xx.reshape(-1, 1)

Xtr, Xva, Xte, ytr, yva, yte = train_val_test_split(XX, yy)
print("train / val / test sizes:", Xtr.shape[0], Xva.shape[0], Xte.shape[0])

The weakness of a single validation set: the score depends on *which* points
happened to land in it, and we "waste" those points by never training on them.
With small datasets that is costly. The standard cure is **cross-validation**.

## 7. k-fold cross-validation (from scratch)

**k-fold cross-validation** uses *all* the data for both training and validation,
fairly:

1. Shuffle, then split the data into **k** equal **folds**.
2. For each fold $j$: train on the other $k-1$ folds, validate on fold $j$.
3. **Average** the $k$ validation scores.

Every point is used for validation exactly once and for training $k-1$ times. The
averaged score is a far more **stable** estimate than a single split. Let's
implement it and use it to **pick the polynomial degree**.

In [ ]:
def k_fold_indices(n, k, rng=rng):
    """Yield (train_idx, val_idx) pairs for k folds."""
    idx = np.arange(n)
    rng.shuffle(idx)
    folds = np.array_split(idx, k)            # k roughly-equal chunks
    for j in range(k):
        val_idx = folds[j]
        train_idx = np.concatenate([folds[i] for i in range(k) if i != j])
        yield train_idx, val_idx

def cv_score_poly(x, y, degree, k=5, rng=rng):
    """Average validation MSE of a degree-`degree` polynomial over k folds."""
    scores = []
    for train_idx, val_idx in k_fold_indices(len(x), k, rng=rng):
        c = np.polyfit(x[train_idx], y[train_idx], degree)   # fit on k-1 folds
        pred = np.polyval(c, x[val_idx])                     # predict held-out fold
        scores.append(mse(y[val_idx], pred))
    return np.mean(scores)                                   # average the folds

In [ ]:
# use cross-validation to choose the best polynomial degree
degrees = range(1, 11)
cv_errs = [cv_score_poly(xx, yy, deg, k=5,
                         rng=np.random.default_rng(deg))   # fresh rng per degree
           for deg in degrees]

best_deg = list(degrees)[int(np.argmin(cv_errs))]
print("best degree by 5-fold CV:", best_deg)

plt.plot(list(degrees), cv_errs, "o-")
plt.axvline(best_deg, color="gray", ls="--", label=f"best deg = {best_deg}")
plt.xlabel("polynomial degree"); plt.ylabel("mean CV MSE")
plt.title("Hyperparameter selection by cross-validation")
plt.legend(); plt.grid(True)
plt.show()

Once cross-validation has chosen the degree, we **retrain on all the
train+validation data** at that degree, and only *then* report the **test** error
as our final honest estimate. We never used the test set to make any choice.

### Optional: the same idea in scikit-learn

`scikit-learn` packages all of this. Our from-scratch versions match its
behaviour; the library just saves typing. (Skipped silently if not installed.)

In [ ]:
try:
    from sklearn.model_selection import cross_val_score
    from sklearn.linear_model import LinearRegression
    from sklearn.preprocessing import PolynomialFeatures
    from sklearn.pipeline import make_pipeline

    model = make_pipeline(PolynomialFeatures(best_deg), LinearRegression())
    # sklearn maximizes score, so it returns NEGATIVE MSE; flip the sign
    neg_mse = cross_val_score(model, xx.reshape(-1, 1), yy,
                              cv=5, scoring="neg_mean_squared_error")
    print("sklearn 5-fold mean MSE:", round(-neg_mse.mean(), 4))
except Exception as e:
    print("scikit-learn not available, skipping:", e)

## Recap

- **Accuracy lies** on imbalanced data — a do-nothing model can score 99%.
- The **confusion matrix** (TP / FP / FN / TN) separates the *types* of error.
- **Precision** = $TP/(TP+FP)$ (punishes false alarms); **recall** =
  $TP/(TP+FN)$ (punishes misses); **F1** is their harmonic mean.
- The **decision threshold** trades precision against recall; the **ROC curve**
  (and its **AUC**) summarizes that trade-off across all thresholds.
- Regression metrics: **MSE**, **RMSE** (same units), **MAE** (robust), and
  **R²** (variance explained; mean-predictor = 0).
- Split **train / validation / test**; pick hyperparameters with **k-fold
  cross-validation**, and touch the **test set only once**.

---
## 📝 Homework (no solutions provided)

1. **Specificity & balanced accuracy.** Write `specificity = TN / (TN + FP)`
   from the confusion counts. Then compute **balanced accuracy** =
   (recall + specificity) / 2 for the lazy "always healthy" model of Section 1.
   Why is balanced accuracy a better one-number summary than plain accuracy here?

2. **The $F_\beta$ score.** The general $F_\beta = (1+\beta^2)\dfrac{P\cdot R}
   {\beta^2 P + R}$ weights recall $\beta$ times as much as precision. Implement
   it and confirm $F_1$ matches your Section 3 result. Compute $F_2$ (recall-
   heavy) and $F_{0.5}$ (precision-heavy) for the Section 4 data at $t = 0.5$.

3. **Pick a threshold for a goal.** Using the Section 4 scored data, find the
   smallest threshold $t$ that achieves **precision ≥ 0.9**, and report the recall
   you get there. (Sweep `thresholds` and check the precision array.)

4. **CV variance.** Run your 5-fold `cv_score_poly` for degree 4 using **ten
   different shuffles** (ten different rng seeds) and print the mean and standard
   deviation of the ten CV scores. Then repeat with **k = 10** folds. Does more
   folds make the estimate more stable?